<div align="center">

### **1.5 Why Monte Carlo**

The same price can be obtained in three equivalent ways: by solving the Black–Scholes equation, by
evaluating the risk-neutral expectation, or by backward induction on a binomial tree, which converges to
both as the time step shrinks (Lecture 3, pp.84–87). For a vanilla call and put the expectation can be
evaluated in closed form, so the Black–Scholes values of §1.4 serve as an exact benchmark.

Finite-difference
methods solve the equation on a grid and are efficient for a single underlying, but their cost grows
rapidly with the number of underlyings. Monte Carlo simulation estimates the expectation directly, and it
earns its place where the other routes break down or become impractical:

- **Path-dependent payoffs.** For conditional barriers (knock-in and knock-out options), corridor options
  and variance or volatility swaps, we need the entirety of the path: whether the stock price stayed within
  the contract's conditions, when it exited them, or the variance realised along the way. Asian options
  depend on the average price over every observation date. Continuous conditions have to be approximated
  on a discrete grid, which can be handled through the contract itself (for example, by tracking end-of-day
  closes or standardised cut-off times and fitting $\delta t$ accordingly) or by a continuity correction to
  the barrier (Broadie, Glasserman and Kou, 1997). All of these products require a method robust enough to
  approximate the payoff across the whole lifespan of the contract.

- **Many contracts on the same underlyings.** The simulated paths depend only on the underlying's dynamics
  ($S_0$, $r$, $\sigma$), not on the contract, so one set of paths prices every option on that underlying,
  across strikes, expirations and product types. This makes Monte Carlo very efficient for a portfolio of
  options, and even more so with several underlyings, where finite differences become infeasible (§1.7).

- **Models without a closed form.** Changing the model, for example to stochastic volatility, changes only
  the simulation step: Euler–Maruyama and Milstein (§1.8) discretise any stochastic differential equation.
  The pricing method itself does not depend on the model; the risk-neutral valuation formula is
  "relatively model-independent", whereas the Black–Scholes formula is not (Lecture 3, p.85).

- **The whole payoff distribution, not just its mean.** The simulated payoffs give the full distribution of
  outcomes, and the estimate reports its own uncertainty: the standard error $\hat s/\sqrt N$ gives a
  confidence interval around the price (§1.6). That isn't better than zero error, but it tells you how far
  to trust the number.

The latter three are very important for managing a large portfolio of options and its risk: simulated
paths turn naturally into risk measures such as maximum drawdown, value-at-risk and expected shortfall
(simulated under the real-world measure, rather than $\mathbb{Q}$, for that purpose).

Here the method is applied to vanilla options, whose price is known in closed form. Pricing contracts that
have a known closed form isolates the method's errors, so they can be measured exactly.

</div>


<div align="center">

### **1.6 The Monte Carlo method**

Monte Carlo pricing follows directly from the valuation formula (Lecture 4, p.5; Wilmott, 2007, ch. 29):

1. Simulate the risk-neutral random walk for $S$ from today's value $S_0$ to expiry $T$. This gives one
   realisation of the stock price path.
2. Calculate the option payoff for this realisation.
3. Repeat for many more realisations.
4. Average the payoffs over all realisations.
5. Discount the average at the risk-free rate. This is the option value.

Whichever scheme generates $S_T$ (§1.8), Monte Carlo draws $N$ independent paths, computes the payoff on each,
and averages and discounts:

$$
\hat V_N = e^{-r(T-t)}\,\frac{1}{N}\sum_{i=1}^{N} \mathrm{Payoff}\big(S_T^{(i)}\big),
\qquad
\mathrm{SE} = \frac{\hat s}{\sqrt N},
$$

where $\hat s$ is the sample standard deviation of the discounted payoffs. By the law of large numbers
$\hat V_N \to V$ as $N \to \infty$, and by the central limit theorem the sampling error falls as
$O(N^{-1/2})$ (Lecture 4, p.9). The closed form evaluates the integral over every possible terminal price;
Monte Carlo estimates it from a random sample.

Every simulation uses the risk-neutral drift $r$, not the real-world drift $\mu$. Simulating with $\mu$
would give the expected payoff in the real world, which is not the price, because the hedging argument of
§1.2 removes $\mu$ from the value.

The random numbers come from a seeded generator,
`numpy.random.default_rng(seed)`, so that every result in this report can be reproduced exactly.

</div>


<div align="center">

### **1.7 Error and computational cost**

#### **Sources of error**

The Monte Carlo estimate has two sources of error (Lecture 4, p.9; Wilmott, 2007, ch. 29):

- **Time discretisation:** the simulation replaces continuous time by finite steps of size $\delta t$.
  For the vanilla prices under geometric Brownian motion studied here, Euler and Milstein have weak error
  of $O(\delta t)$. Approximating a continuously monitored barrier can converge more slowly. For path-dependent products, such as options with continuous
  knock-in/knock-out barriers, it can also miss payoff-altering events that occur between steps: for
  example, an intraday stock price that breaches a knock-out barrier and recovers by the close, when
  $\delta t$ is one day and payoffs are calculated from end-of-day prices.

- **A finite number of realizations**, of $O(N^{-1/2})$: we simulate only $N$ of the infinitely many
  possible paths of the asset price to $T$. This error is measured by the standard error of §1.6.

</div>


<div align="center">

#### **Computational cost**

The total number of calculations budgeted for the simulation is the number of paths times the number of
steps per path, $K_{\text{Compute}} = N/\delta t$. Any budget spent on more paths reduces the budget for
finer time steps, and vice versa.

The total error is of the order of the larger of the two sources,
$\delta t$ and $N^{-1/2}$. Setting the two errors equal, $\delta t = N^{-1/2}$, and substituting into the
budget gives $K_{\text{Compute}} = N/\delta t = N^{3/2}$, so

$$
N = K_{\text{Compute}}^{2/3}, \qquad \delta t = K_{\text{Compute}}^{-1/3}, \qquad
\text{error} \approx \delta t = K_{\text{Compute}}^{-1/3}.
$$

For example, a budget of $K_{\text{Compute}} = 10^6$ simulated steps has the illustrative scaling allocation $N = 10^4$ paths with
$\delta t = 10^{-2}$. This balances orders only; the measured error constants determine the practical allocation in §2.2.

Accuracy is therefore expensive: halving the error means halving both sources, which
needs twice as many steps and four times as many paths, or eight times the work.

The exact scheme, however,
has no time-step error, so for a European payoff a single step suffices and the entire budget goes to $N$:
halving the error then needs only four times as many paths (§2.2 includes an allocation based on measured error constants).

Further, Monte Carlo is relatively inexpensive computationally compared with finite-difference methods
when there are more than about three dimensions (underlyings). The brunt of the computation is path
generation, and the same set of simulated paths can price every option in the book through its time steps.
$K_{\text{Compute}}$ scales linearly with $N$, $1/\delta t$ and the number of underlyings $d$: each
additional underlying adds one more set of paths, and everything after that (the payoff calculations) is
cheap.

Finite-difference and other grid methods, on the other hand, have to work out the option's value at
every combination of prices of the underlyings, so their cost grows exponentially and becomes infeasible
beyond a handful of underlyings.

</div>


<div align="center">

#### **Drawbacks**

Monte Carlo is slow to converge: its error falls only as $N^{-1/2}$, so each extra decimal place of
accuracy costs a hundred times as many paths. The greeks also need extra work, because the method returns a
price at a single point rather than its derivatives (Lecture 4, pp.12–14).

**Early exercise** is functionally very difficult to account for (Lecture 4, p.14). Monte Carlo is an
inherently forward-looking method: at any point $(S_t, t)$ on a path, deciding whether to exercise requires
the continuation value, the expected discounted value of holding on, which depends on future payoffs that
cannot be estimated at time $t$, at least not well and not over a decent time horizon. Using the path's own
future instead would look ahead, and overvalue the option.

The early-exercise constraint itself is known,
$V(S,t) \ge \mathrm{Payoff}(S)$: the option should be exercised wherever the continuation value falls below
the immediate payoff (Wilmott, 2007, §29.15). Checking it means revaluing the option at every point in
asset–time space that the paths pass through, essentially starting a new simulation in a loop at each one,
and the compute and time taken grow exponentially with the number of exercise dates. Finite-difference
methods, which work backwards through every $(S, t)$, handle this naturally.

That said, the least-squares
regression technique of Longstaff and Schwartz (2001), which estimates the continuation value by regressing
discounted future cash flows on the current stock price across all paths, holds up well to this issue.

More
generally, Monte Carlo simulations are very flexible and configurable techniques that retain their
robustness because of the large sample: by the central limit theorem the error of the estimate is
approximately normal whatever the payoff, so its standard error remains a reliable guide.

For the European options in this report early exercise does not arise, so the drawbacks that matter here
are the speed of convergence and the greeks. They motivate the simulation schemes and the variance
reduction that follow (§1.8–1.9), and the treatment of the greeks in the results.

</div>


<div align="center">

### **1.8 Simulation schemes**

Step 1 of the algorithm needs values of $S$ along each path. Divide $[0, T]$ into $n$ steps of size $\delta t = T/n$, and let
$\phi_1, \phi_2, \dots$ be independent standard normal draws. The Brownian increment over one step is then
$\delta X = \sqrt{\delta t}\,\phi$, which has mean $0$ and variance $\delta t$. The exam asks for three schemes.

#### **Exact simulation**

Under $\mathbb{Q}$, $\ln S_T$ is normally distributed, so $S_T$ can be
generated exactly from a single standard normal draw (Lecture 4, p.7):

$$
\ln S_T \sim N\!\left(\ln S + (r-\tfrac12\sigma^2)(T-t),\; \sigma^2(T-t)\right)

$$

Equivalently,

$$
S_T = S\,\exp\!\left((r-\tfrac12\sigma^2)(T-t) + \sigma\sqrt{T-t}\,\phi\right),\quad \phi \sim N(0,1).
$$

This is the closed-form solution for simulating the stock price named in the exam brief. Over one step
of any size, the same result reads

$$
S_{n+1} = S_n \exp\!\left(\left(r - \tfrac12\sigma^2\right)\delta t + \sigma\sqrt{\delta t}\;\phi_{n+1}\right).
$$

This has no discretisation error for any $\delta t$. A European payoff depends only on $S_T$, so the whole
interval can be covered in a single step with $\delta t = T$ (Lecture 4, p.8). Smaller steps are needed only
for path-dependent payoffs, which are monitored along the path.

#### **One step or 252?**

For a European payoff the time steps of the exact scheme are not value-additive.
Summing 252 independent increments, each normal with variance $\sigma^2\delta t$, gives $\ln S_T$ exactly the
same distribution as a single increment of variance $\sigma^2 T$, so the extra steps introduce no drift and
no additional noise; a one-step and a 252-step estimate differ only because they use different random draws.

All else equal, the only effect of the extra steps is cost: at a fixed computing budget $K_{\text{Compute}}$,
252 steps leave 252 times fewer paths, and a standard error $\sqrt{252} \approx 16$ times larger.

The
252-step exact paths still earn their place for two reasons. Path-dependent payoffs need the intermediate
prices, and there more steps *reduce* the error from monitoring a continuous condition at discrete times.
They also provide the exact reference path, driven by the same random draws, against which the Euler and
Milstein paths are measured.

</div>


<div align="center">

#### **Convergence of the numerical schemes**

Both are numerical schemes for simulating a stochastic process along each
path, when the pricing functional needs the process at discrete times and the stochastic differential
equation cannot be integrated exactly (Jäckel, 2002, ch. 4, pp.31–33; Kloeden and Platen, 1992). For a
general equation $dS = a(S)\,dt + b(S)\,dX$, a scheme is judged by how its approximation $\hat S$ converges
as the step size $\delta t$ shrinks:

- **Strong convergence** requires the numerical approximation to a given path to converge to the exact
  solution along that same path. Driven by the same Brownian increments, the simulated value approaches
  the true one: $\mathbb{E}\,\big|\hat S_T - S_T\big| = O(\delta t^{\gamma})$.
- **Weak convergence** only requires the outputs of the model to converge. Expectations of functions of the
  path, such as an option price, approach their true values,
  $\big|\mathbb{E}[f(\hat S_T)] - \mathbb{E}[f(S_T)]\big| = O(\delta t^{\beta})$, even if individual paths do not.

</div>


<div align="center">

#### **Euler–Maruyama**

Euler–Maruyama is the simplest step beyond the general form: it freezes the coefficients at the start
of each step and replaces $dX$ by $\delta X = \sqrt{\delta t}\,\phi$ (Lecture 4, p.6; Wilmott, 2007, ch. 29),

$$
\hat S_{n+1} = \hat S_n + a(\hat S_n)\,\delta t + b(\hat S_n)\,\sqrt{\delta t}\;\phi_{n+1}.
$$

For geometric Brownian motion this is a multiplicative update,
$\hat S_{n+1} = \hat S_n\left(1 + r\,\delta t + \sigma\sqrt{\delta t}\,\phi_{n+1}\right)$ (with a dividend yield $D$,
$r$ would be replaced by $r - D$; here $D = 0$). It has strong convergence of order $\tfrac12$ and weak
convergence of order 1.

</div>


<div align="center">

#### **Milstein**

Milstein is more accurate pathwise. It adds the next term of the Itô–Taylor expansion of
$dS = a\,dt + b\,dX$, the one in $(\delta X)^2$, which is of the same order as $\delta t$ and so cannot be dropped:

$$
\hat S_{n+1} = \hat S_n + a\,\delta t + b\,\delta X + \tfrac12\,b\,b'\left((\delta X)^2 - \delta t\right).
$$

For geometric Brownian motion ($b = \sigma S$, $b' = \sigma$) this gives
$\hat S_{n+1} = \hat S_n\left(1 + r\,\delta t + \sigma\sqrt{\delta t}\,\phi_{n+1} + \tfrac12\sigma^2\delta t\,(\phi_{n+1}^2 - 1)\right)$.
It has strong convergence of order 1 and weak convergence of order 1.

Option pricing needs only weak convergence, where the two schemes have the same order, so Milstein is not
expected to price much more accurately than Euler, even though its paths are much closer to the exact ones.
Jäckel notes, however, that empirically, schemes with stronger convergence tend to perform better for option
pricing. Driving all three schemes with the same draws $\phi$ allows both kinds of error to be measured
directly against the exact solution, and §2.2 tests which effect dominates here.

</div>


<div align="center">

### **1.9 Variance reduction: antithetic variates**

The $N^{-1/2}$ rate cannot be changed, so the sampling error can only be reduced by simulating more paths
or by reducing the variance of the payoffs (Wilmott, 2007, ch. 29; Jäckel, 2002, ch. 10, pp.111–112). Antithetic
variates make use of the fact that every drawn path has a mirror image, driven by the same Gaussian variate
vector with its sign reversed, $-\phi$. The standard normal distribution is symmetric, so $-\phi$ is as valid
a draw as $\phi$: each pair of paths is simulated from a single set of random numbers, and the two
discounted payoffs $Y$ are averaged,

$$
\hat V_N^{\text{AV}} = \frac{1}{N/2}\sum_{i=1}^{N/2} \tfrac12\left[\,Y\big(\phi^{(i)}\big) + Y\big(-\phi^{(i)}\big)\right],
$$

where $N$ counts every simulated path. This is a cheap way to reduce the variance of the estimate, since
the random numbers are drawn once for two paths, but it is not a free doubling of the sample: both paths
still have to be built and priced, and the two members of a pair are not independent of each other.

The
central limit theorem still applies, because we count the pairwise average as a single sample and the
pairs are independent of one another; the standard error is therefore computed from the $N/2$ pair
averages, not from the $N$ individual payoffs. Using
$\mathrm{Var}[X+Y] = \mathrm{Var}[X] + 2\,\mathrm{Cov}[X,Y] + \mathrm{Var}[Y]$ (Tutorial JU26T4, p.6), the
variance of each pair average is

$$
\mathrm{Var}\!\left[\tfrac12\big(Y(\phi) + Y(-\phi)\big)\right]
= \tfrac12\Big(\mathrm{Var}[Y] + \mathrm{Cov}\big[Y(\phi),\,Y(-\phi)\big]\Big).
$$

</div>


<div align="center">

#### **When antithetic pairing helps**

Without the covariance term this would equal the variance of the average of two independent paths, and
nothing would be gained: the method works only when $Y(\phi)$ and $Y(-\phi)$ are negatively correlated.

Mirroring forces the sample mean of the draws to be exactly zero, correcting their first moment, so it works
especially well when the payoff depends mainly on that first moment, the direction of the draws. A call or
put payoff is monotone in $S_T$, and $S_T$ increases with $\phi$, so the covariance is negative and the
estimator has a lower variance than $N$ independent paths at the same cost. A payoff that depends on the
size of the move rather than its direction, such as a straddle, gains little or nothing.

The method is most useful with ordinary pseudo-random numbers, as used here, whose sample moments fluctuate
from run to run. Low-discrepancy (Sobol) sequences are already evenly balanced, so mirroring them has little
left to correct and can interfere with the structure that makes them work; the two are best not combined
(Jäckel, 2002, pp.111–112).

</div>
